# Guía de Estudio Extra – Parcial 2 Práctico
## ISIS-2611 | Más ejercicios de "Encuentra el Error"

Este notebook complementa la guía principal. Contiene ejercicios adicionales organizados por tema.

| # | Tema | Subtema |
|---|------|---------|
| 1-6 | MLP/Feed Forward | Shapes, overfitting, learning rate, callbacks |
| 7-13 | CNN | Strides, padding, transfer learning, augmentación |
| 14-20 | RNN/LSTM/GRU | return_sequences, reshape, bidireccional, time series |
| 21-27 | Embeddings/NLP | Tokenización, BERT, truncado, padding |
| 28-33 | Bayes | Smoothing, probabilidades, independencia |
| 34-40 | Clustering | DBSCAN fino, jerárquico, interpretación |
| 41-48 | Métricas y Evaluación | Overfitting, split, imbalance, confusion matrix |
| 49-55 | Pipeline/Preprocesamiento | Encoders, SMOTE, imputación, leakage |


---
# BLOQUE 1 – MLP / Feed Forward

## Ejercicio 1.1 – Número incorrecto de neuronas en la salida
**Caso**: Clasificar imágenes de ropa en 10 categorías (Fashion MNIST).

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='softmax')   # ERROR: solo 1 neurona para 10 clases
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: Dense(1, activation='softmax') para 10 clases
#   Con 1 sola neurona y softmax, la salida siempre será 1.0 (trivial).
#   Para N clases necesitas N neuronas.
#   El error de shape aparece cuando calculas la pérdida con 10 etiquetas distintas.
#
# REGLA: capa de salida → tantas neuronas como clases.

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')  # ✅ 10 neuronas para 10 clases
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


---
## Ejercicio 1.2 – Learning rate demasiado alto
**Caso**: El modelo de regresión no converge, la loss oscila o explota.

In [ ]:
# ❌ CÓDIGO CON ERRORES
import tensorflow as tf

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dense(1)
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=10.0),  # ERROR: lr absurdamente alto
    loss='mse'
)

history = model.fit(X_train, y_train, epochs=50)
# La loss imprime: nan, nan, nan...


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: learning_rate=10.0
#   El gradiente se multiplica por 10 en cada paso.
#   Los pesos se disparan, produciendo NaN (exploding gradients).
#
# VALORES TÍPICOS:
#   Adam:    lr = 0.001  (default recomendado)
#   SGD:     lr = 0.01
#   Para fine-tuning de modelos pre-entrenados: lr = 0.00001
#
# SÍNTOMAS de lr muy alto: loss = NaN, loss oscila enormemente
# SÍNTOMAS de lr muy bajo: loss baja muy despacio, necesita muchas épocas

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # ✅
    loss='mse'
)


---
## Ejercicio 1.3 – Datos de validación usados para entrenar
**Caso**: Se quiere evaluar el rendimiento real del modelo en datos no vistos.

In [ ]:
# ❌ CÓDIGO CON ERRORES
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test)   # ERROR: el test set se usa como validación
)

# Luego reportan el val_accuracy del entrenamiento como resultado final
# 'Nuestro modelo tiene 92% de accuracy' ← pero ese es el val visto durante el entrenamiento


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: usar el test set como validation_data durante el entrenamiento
#   Aunque no se ajustan los pesos con ese conjunto, se puede usar para
#   selección de hiperparámetros (early stopping, arquitectura), lo que
#   filtra información del test set al proceso de diseño del modelo.
#
# CORRECCIÓN: usar un conjunto de validación SEPARADO del test set.

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, random_state=42)

model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val)      # ✅ validación separada del test
)

# Evaluación FINAL con test (solo una vez al final)
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Resultado final en test set: {test_acc:.4f}')


---
## Ejercicio 1.4 – Overfitting sin regularización
**Caso**: El modelo tiene 99% accuracy en entrenamiento y 55% en test.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = keras.Sequential([
    layers.Dense(2048, activation='relu', input_shape=(50,)),  # ERROR: demasiadas neuronas
    layers.Dense(2048, activation='relu'),
    layers.Dense(2048, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=200)  # ERROR: demasiadas épocas sin early stopping

# train_acc = 0.99, test_acc = 0.55 → overfitting severo


In [ ]:
# ✅ SOLUCIÓN
#
# PROBLEMAS:
#   1. Modelo demasiado grande para los datos (capacidad >> necesidad)
#   2. Sin Dropout ni regularización
#   3. Sin EarlyStopping → el modelo memoriza el train set
#
# SEÑALES de overfitting:
#   - train_loss baja, val_loss sube (en curvas de aprendizaje)
#   - gran brecha entre train_acc y val_acc

from tensorflow.keras import callbacks, regularizers

model = keras.Sequential([
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001),
                 input_shape=(50,)),
    layers.Dropout(0.3),                   # ✅ regularización
    layers.Dense(64, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True              # ✅ restaura el mejor modelo
)

model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2,
    callbacks=[early_stop]                 # ✅ para automáticamente
)


---
## Ejercicio 1.5 – Batch Normalization en lugar incorrecto
**Caso**: Arquitectura con BatchNormalization mal ubicada.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = keras.Sequential([
    layers.BatchNormalization(input_shape=(100,)),  # ERROR: BN antes de la primera Dense
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
])
# La BN antes de cualquier Dense no tiene el efecto esperado.
# También: BN justo antes de la capa de salida altera las logits.


In [ ]:
# ✅ SOLUCIÓN
#
# BatchNormalization va DESPUÉS de Dense y ANTES de la activación (o después, hay debate),
# pero NUNCA justo antes de la capa de salida.
# El orden más común en práctica: Dense → BN → Activation → Dropout

model = keras.Sequential([
    layers.Dense(256, input_shape=(100,)),
    layers.BatchNormalization(),           # ✅ después de Dense
    layers.Activation('relu'),
    layers.Dropout(0.3),

    layers.Dense(128),
    layers.BatchNormalization(),           # ✅
    layers.Activation('relu'),
    layers.Dropout(0.2),

    layers.Dense(10, activation='softmax') # ✅ sin BN antes de la salida
])


---
## Ejercicio 1.6 – Usar model.predict mal para clasificación
**Caso**: Quieren saber la clase predicha para cada ejemplo.

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Modelo de clasificación multiclase (10 clases, softmax)
y_pred = model.predict(X_test)     # Esto devuelve probabilidades, no clases

# Luego hacen:
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)  # ERROR: y_pred tiene shape (N, 10), no (N,)


In [ ]:
# ✅ SOLUCIÓN
#
# model.predict() devuelve probabilidades (softmax output), shape (N, 10).
# Para obtener la clase hay que tomar el argmax.

import numpy as np
from sklearn.metrics import accuracy_score, classification_report

y_prob  = model.predict(X_test)              # shape (N, 10) → probabilidades
y_pred  = np.argmax(y_prob, axis=1)         # ✅ shape (N,) → clase predicha

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

# Para clasificación binaria (sigmoid):
# y_prob = model.predict(X_test)            # shape (N, 1)
# y_pred = (y_prob > 0.5).astype(int).flatten()   # ✅ umbral 0.5


---
# BLOQUE 2 – CNN

## Ejercicio 2.1 – Padding incorrecto hace que el feature map desaparezca
**Caso**: CNN para imágenes pequeñas (8×8), después de 3 capas conv la imagen tiene tamaño 0.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow.keras import layers, models

# Imagen de entrada: 8x8x1
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(8,8,1)),  # 8→6
    layers.Conv2D(64, (3,3), activation='relu'),                        # 6→4
    layers.Conv2D(128,(3,3), activation='relu'),                        # 4→2
    layers.MaxPooling2D((2,2)),                                          # 2→1
    layers.Conv2D(256,(3,3), activation='relu'),  # ERROR: 1x1 con kernel 3x3 → FALLA
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: padding='valid' (default) reduce el tamaño H y W en cada capa Conv.
#   Con imágenes pequeñas, se quedan sin espacio para el kernel.
#
# OPCIONES:
#   1. Usar padding='same' → mantiene H y W constantes
#   2. Reducir el número de capas Conv
#   3. Usar imágenes más grandes (hacer resize)
#
# padding='valid': sin relleno, el tamaño se reduce. Fórmula: out = (in - kernel) / stride + 1
# padding='same':  con relleno de ceros, el tamaño se mantiene (stride=1)

model = models.Sequential([
    layers.Conv2D(32,  (3,3), padding='same', activation='relu', input_shape=(8,8,1)),
    layers.Conv2D(64,  (3,3), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2)),   # 8→4
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),   # ✅ alternativa a Flatten, no depende del tamaño
    layers.Dense(10, activation='softmax')
])


---
## Ejercicio 2.2 – Transfer learning: no congelar el modelo base
**Caso**: Se usa MobileNetV2 pre-entrenado para clasificar 5 tipos de flores con pocos datos.

In [ ]:
# ❌ CÓDIGO CON ERRORES
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# ERROR: no se congela el modelo base
# base_model.trainable = False  ← FALTA

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(5, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy')
model.fit(X_train, y_train, epochs=10)
# Resultado: el modelo sobreajusta o destruye los pesos de ImageNet


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: no congelar el modelo base en transfer learning
#   Con pocos datos, si dejamos los millones de parámetros de MobileNetV2 entrenables,
#   el modelo sobreajusta a los pocos ejemplos de flores y destruye los pesos de ImageNet.
#
# FLUJO CORRECTO DE TRANSFER LEARNING:
#   Fase 1 – Feature extraction: congelar base, entrenar solo el 'top'
#   Fase 2 – Fine-tuning (opcional): descongelar capas superiores con lr muy bajo

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False    # ✅ CONGELAR el modelo base

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(5, activation='softmax')
])
# lr más pequeño para no sobreescribir los pesos aprendidos
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


---
## Ejercicio 2.3 – Data Augmentation aplicado al test set
**Caso**: Se preparan generadores de imágenes para entrenamiento y test.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Misma configuración para train Y test
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory('data/train', target_size=(150,150))
test_gen  = datagen.flow_from_directory('data/test',  target_size=(150,150))  # ERROR


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: aplicar data augmentation (rotaciones, flips, zooms) al conjunto de TEST
#   El augmentation sirve para artificialmente ampliar el conjunto de ENTRENAMIENTO.
#   En TEST queremos evaluar con imágenes reales tal como llegan, sin modificarlas.
#   Solo se aplica rescale (normalización) al test set.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)  # ✅ augmentation solo en train

test_datagen = ImageDataGenerator(rescale=1./255)  # ✅ solo normalizar en test

train_gen = train_datagen.flow_from_directory('data/train', target_size=(150,150))
test_gen  = test_datagen.flow_from_directory('data/test',   target_size=(150,150))


---
## Ejercicio 2.4 – GlobalAveragePooling vs Flatten: consecuencias
**Caso**: CNN con GlobalAveragePooling seguido de Dense de tamaño incorrecto.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = models.Sequential([
    layers.Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128,(3,3), padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),  # Salida: (128,) — promedia cada mapa
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='sigmoid')   # ERROR: sigmoid para 3 clases
])
model.compile(loss='binary_crossentropy', metrics=['accuracy'])  # ERROR: pérdida binaria


In [ ]:
# ✅ SOLUCIÓN
#
# GlobalAveragePooling2D promedia cada mapa de características → salida (num_filtros,)
# No depende del tamaño espacial de la imagen → es más robusto que Flatten.
#
# ERRORES aquí:
#   1. sigmoid en 3 clases → usar softmax
#   2. binary_crossentropy para 3 clases → usar sparse_categorical_crossentropy

model = models.Sequential([
    layers.Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128,(3,3), padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')              # ✅
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',            # ✅
    metrics=['accuracy']
)


---
## Ejercicio 2.5 – Preprocesamiento incorrecto para modelo pre-entrenado
**Caso**: Se usa VGG16 pero se normaliza con ÷255 en vez del preprocesamiento de VGG.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow.keras.applications import VGG16

base = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Preprocesamiento:
X_train_norm = X_train / 255.0   # ERROR: VGG16 NO fue entrenado con datos en [0,1]
# VGG16 espera píxeles en [0, 255] con media de ImageNet substraída


In [ ]:
# ✅ SOLUCIÓN
#
# Cada arquitectura pre-entrenada tiene su propio preprocesamiento.
# VGG16, ResNet, MobileNet etc. fueron entrenados con estadísticas de ImageNet.
# Usar la función preprocess_input de cada arquitectura.
#
# REGLA: usa SIEMPRE preprocess_input del mismo módulo que el modelo.

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

base = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# ✅ preprocesamiento correcto para VGG16
X_train_prep = preprocess_input(X_train.astype('float32'))   # centra con media de ImageNet

# Para MobileNetV2:
# from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Para ResNet50:
# from tensorflow.keras.applications.resnet50 import preprocess_input


---
## Ejercicio 2.6 – Strides grandes que saltan información
**Caso**: Se quiere mantener la mayor resolución posible en features de bajo nivel.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = models.Sequential([
    layers.Conv2D(32, (3,3), strides=(4,4), activation='relu', input_shape=(64,64,3)),
    # Con strides=(4,4): 64 → 16 de golpe, perdiendo mucho detalle
    layers.Conv2D(64, (3,3), strides=(4,4), activation='relu'),
    # 16 → 4
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])
# La primera capa salta de 4 en 4 píxeles → se pierde información de bordes finos


In [ ]:
# ✅ SOLUCIÓN
#
# STRIDES controla cuánto se mueve el filtro en cada paso.
#   strides=(1,1): el filtro se mueve de 1 en 1 (máximo detalle, por defecto)
#   strides=(2,2): reduce a la mitad (como MaxPooling pero entrenado)
#   strides=(4,4): muy agresivo, se pierde mucha información espacial
#
# CUÁNDO usar strides > 1:
#   En la primera capa de redes profundas (ResNet) para imágenes grandes (224x224)
#   Cuando memoria/velocidad es prioritaria sobre resolución
#
# MEJOR para imágenes pequeñas: strides=(1,1) + MaxPooling para reducción

model = models.Sequential([
    layers.Conv2D(32, (3,3), strides=(1,1), padding='same', activation='relu', input_shape=(64,64,3)),
    layers.MaxPooling2D((2,2)),   # 64→32
    layers.Conv2D(64, (3,3), strides=(1,1), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2)),   # 32→16
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])


---
# BLOQUE 3 – RNN / LSTM / GRU

## Ejercicio 3.1 – return_sequences incorrecto al apilar LSTMs
**Caso**: Se quieren apilar dos capas LSTM.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.LSTM(64),                           # ERROR: return_sequences=False por defecto
    layers.LSTM(32),                           # No puede recibir un vector, necesita secuencia
    layers.Dense(1, activation='sigmoid')
])
# Error: Input 0 of layer lstm_1 is incompatible with the layer:
# expected ndim=3, found ndim=2


In [ ]:
# ✅ SOLUCIÓN
#
# return_sequences=False (default): devuelve solo el último estado → shape (batch, units)
# return_sequences=True:            devuelve todos los estados → shape (batch, timesteps, units)
#
# REGLA:
#   - Si el LSTM va seguido de OTRO LSTM → return_sequences=True
#   - Si el LSTM va seguido de Dense     → return_sequences=False (o usar el último output)

model = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.LSTM(64, return_sequences=True),    # ✅ devuelve secuencia para el siguiente LSTM
    layers.LSTM(32, return_sequences=False),   # ✅ último LSTM → solo el último estado
    layers.Dense(1, activation='sigmoid')
])


---
## Ejercicio 3.2 – Series de tiempo: datos sin reshape correcto
**Caso**: Predecir el precio de una acción usando los últimos 30 días.

In [ ]:
# ❌ CÓDIGO CON ERRORES
import numpy as np

# precios: array de 1000 días
# Se hacen ventanas de 30 días
X = np.array([precios[i:i+30] for i in range(len(precios)-30)])
y = np.array([precios[i+30]   for i in range(len(precios)-30)])

print(X.shape)  # (970, 30)  ← 2D, sin dimensión de features

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30,)),   # ERROR: LSTM espera (timesteps, features)
    layers.Dense(1)
])


In [ ]:
# ✅ SOLUCIÓN
#
# LSTM espera input de forma (batch, timesteps, features).
# Con una sola variable (precio), features=1.
# Hay que hacer reshape explícito: (N, 30) → (N, 30, 1)

X = np.array([precios[i:i+30] for i in range(len(precios)-30)])
X = X.reshape(X.shape[0], X.shape[1], 1)  # ✅ (970, 30, 1)

y = np.array([precios[i+30] for i in range(len(precios)-30)])

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30, 1)),  # ✅ (timesteps=30, features=1)
    layers.Dense(1)                        # regresión → sin activación
])
model.compile(optimizer='adam', loss='mse')


---
## Ejercicio 3.3 – GRU con Dense directo al tener return_sequences=True
**Caso**: GRU para clasificar géneros musicales.

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = keras.Sequential([
    layers.Embedding(5000, 32, input_length=100),
    layers.GRU(64, return_sequences=True),  # Devuelve shape (batch, 100, 64)
    layers.Dense(8, activation='softmax')   # ERROR: Dense espera 2D, recibe 3D
])
# Error: Dense recibe un tensor 3D (batch, 100, 64) y aplica Dense a cada timestep.
# Tendrá 100 × 8 salidas, no una sola predicción por secuencia.


In [ ]:
# ✅ SOLUCIÓN – opción A: quitar return_sequences
model_a = keras.Sequential([
    layers.Embedding(5000, 32, input_length=100),
    layers.GRU(64, return_sequences=False),  # ✅ shape (batch, 64)
    layers.Dense(8, activation='softmax')
])

# ✅ SOLUCIÓN – opción B: agregar GlobalAveragePooling o Flatten entre GRU y Dense
model_b = keras.Sequential([
    layers.Embedding(5000, 32, input_length=100),
    layers.GRU(64, return_sequences=True),
    layers.GlobalAveragePooling1D(),         # ✅ promedia los timesteps → (batch, 64)
    layers.Dense(8, activation='softmax')
])


---
## Ejercicio 3.4 – Padding post vs pre: impacto en LSTM
**Caso**: Clasificación de reseñas con longitudes muy variables (10 a 500 palabras).

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow.keras.preprocessing.sequence import pad_sequences

# La mayoría de reseñas son largas; unas pocas son cortas.
X_pad = pad_sequences(sequences, maxlen=200, padding='post', truncating='post')
#                                                              ↑
# padding='post': los ceros se agregan AL FINAL de la secuencia.
# Para un LSTM que lee de izquierda a derecha, los ceros de relleno
# serán los últimos pasos, y el estado final del LSTM estará
# afectado por muchos pasos de 'cero' para secuencias cortas.


In [ ]:
# ✅ SOLUCIÓN
#
# Para LSTM, padding='pre' (al inicio) es mejor:
#   Los ceros quedan al principio y el estado final del LSTM
#   refleja el contenido real de la secuencia (últimas palabras).
#
# CONVENCIÓN:
#   padding='pre'  → agregar ceros al INICIO → mejor para LSTM estándar
#   padding='post' → agregar ceros al FINAL  → necesario para CNN 1D
#
# Aún mejor: usar Masking layer para que la red ignore los ceros.

X_pad = pad_sequences(sequences, maxlen=200, padding='pre', truncating='pre')  # ✅

# Con Masking (ignora ceros automáticamente):
model = keras.Sequential([
    layers.Embedding(VOCAB, 64, input_length=200),
    layers.Masking(mask_value=0),   # ✅ ignora los pasos de padding
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')
])


---
## Ejercicio 3.5 – Bidireccional donde no debería
**Caso**: Generación de texto carácter por carácter (el modelo predice el siguiente carácter).

In [ ]:
# ❌ CÓDIGO CON ERRORES
model = keras.Sequential([
    layers.Embedding(100, 32, input_length=50),
    layers.Bidirectional(layers.LSTM(64)),  # ERROR: bidireccional para generación
    layers.Dense(100, activation='softmax')
])
# Para predecir el siguiente token, solo debería ver los anteriores.
# Bidireccional ve el futuro, lo que es trampa en generación de texto.


In [ ]:
# ✅ SOLUCIÓN
#
# BIDIRECCIONAL:
#   Lee la secuencia de izquierda a derecha Y de derecha a izquierda.
#   Tiene acceso a contexto FUTURO y PASADO.
#
# ✅ ÚSALO EN: clasificación de texto, NER, análisis de sentimientos
#              (donde tienes toda la oración disponible)
#
# ❌ NO LO USES EN: generación de texto, predicción de siguiente token,
#                  predicción de series de tiempo (el futuro no existe todavía)

# Para generación de texto: LSTM unidireccional
model_generacion = keras.Sequential([
    layers.Embedding(100, 32, input_length=50),
    layers.LSTM(64),                    # ✅ solo ve el pasado
    layers.Dense(100, activation='softmax')
])

# Para clasificación de sentimientos: Bidireccional está bien
model_clasificacion = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.Bidirectional(layers.LSTM(64)),   # ✅ ve todo el texto
    layers.Dense(1, activation='sigmoid')
])


---
# BLOQUE 4 – Embeddings / NLP

## Ejercicio 4.1 – One-hot encoding de texto vs Embedding
**Caso**: Clasificar tweets con vocabulario de 50.000 palabras.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Representar cada token como vector one-hot de 50.000 dimensiones
vocab_size = 50000
max_len    = 100

# Para una sola secuencia de 100 tokens:
# X.shape = (100, 50000) → gigantesco y disperso
# Para 10.000 tweets: (10000, 100, 50000) → imposible en RAM

# ERROR: one-hot para vocabularios grandes es inviable y no captura semántica


In [ ]:
# ✅ SOLUCIÓN
#
# ONE-HOT para texto con vocabulario grande:
#   - Dimensionalidad: vocab_size (50.000) por token
#   - Disperso: casi todo es 0
#   - No captura similitud semántica (rey y reina son ortogonales)
#
# EMBEDDING:
#   - Dimensionalidad: embed_dim (64-300) por token → 1000× más compacto
#   - Denso y aprendido: palabras similares tienen vectores cercanos
#   - Permite al modelo aprender relaciones semánticas
#
# CUÁNDO usar one-hot:
#   Vocabularios PEQUEÑOS (< 50 categorías), variables categóricas en ML clásico

VOCAB_SIZE = 50000
EMBED_DIM  = 128    # solo 128 en vez de 50.000
MAX_LEN    = 100

model = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    # Salida: (batch, 100, 128) — densa y semánticamente rica
    layers.GlobalAveragePooling1D(),
    layers.Dense(2, activation='softmax')
])


---
## Ejercicio 4.2 – Tokenizer: fit en test set también
**Caso**: Pipeline de clasificación de tickets.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(textos_train + textos_test)   # ERROR: incluye textos de test

X_train = pad_sequences(tokenizer.texts_to_sequences(textos_train), maxlen=100)
X_test  = pad_sequences(tokenizer.texts_to_sequences(textos_test),  maxlen=100)


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: fit_on_texts con train+test
#   El vocabulario del tokenizer se construye con las frecuencias de TODOS los textos,
#   incluyendo el test. Esto es data leakage conceptual: en producción, el test
#   puede contener palabras nunca vistas durante el entrenamiento.
#
# CORRECCIÓN: fit solo en train. Las palabras del test no vistas → OOV token.

tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')   # ✅ maneja palabras desconocidas
tokenizer.fit_on_texts(textos_train)                        # ✅ solo train

X_train = pad_sequences(tokenizer.texts_to_sequences(textos_train), maxlen=100, padding='pre')
X_test  = pad_sequences(tokenizer.texts_to_sequences(textos_test),  maxlen=100, padding='pre')


---
## Ejercicio 4.3 – TF-IDF antes del split
**Caso**: Pipeline de clasificación de textos con sklearn.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(textos)   # ERROR: fit en todo el corpus

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2)

clf = MultinomialNB()
clf.fit(X_train, y_train)


In [ ]:
# ✅ SOLUCIÓN
#
# TfidfVectorizer aprende IDF (frecuencia inversa de documento) de TODOS los textos.
# El IDF del test set se filtra al train → data leakage.
#
# CORRECCIÓN: usando Pipeline de sklearn que garantiza el orden correcto.

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# ✅ Split primero
X_train_raw, X_test_raw, y_train, y_test = train_test_split(textos, y, test_size=0.2)

# ✅ Pipeline: fit_transform en train, transform en test automáticamente
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf',   MultinomialNB())
])
pipe.fit(X_train_raw, y_train)

print(pipe.score(X_test_raw, y_test))


---
## Ejercicio 4.4 – Embeddings de BERT: dimensión del vector CLS
**Caso**: Se usan embeddings de BERT y se alimentan a un MLP con shape incorrecto.

In [ ]:
# ❌ CÓDIGO CON ERRORES
import torch
from transformers import BertTokenizer, BertModel

tokenizador = BertTokenizer.from_pretrained('bert-base-uncased')
modelo_bert = BertModel.from_pretrained('bert-base-uncased')

def obtener_embedding(texto):
    inputs = tokenizador(texto, return_tensors='pt')
    with torch.no_grad():
        outputs = modelo_bert(**inputs)
    # ERROR: toma toda la secuencia en vez de solo el token [CLS]
    return outputs.last_hidden_state.squeeze(0).numpy()  # shape: (num_tokens, 768)

# Luego intenta meter esto en sklearn con shape inconsistente
X = [obtener_embedding(t) for t in textos]  # lista de arrays con shapes distintos
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(X, y)  # ERROR: sklearn no acepta listas de arrays de distinto tamaño


In [ ]:
# ✅ SOLUCIÓN
#
# Para obtener UN vector por oración con BERT:
#   Opción A: token [CLS] (posición 0) → shape (768,)
#   Opción B: promedio de todos los tokens → shape (768,)
#
# BERT-base devuelve embeddings de 768 dimensiones.
# BERT-large: 1024. GPT-2: 768. etc.

def obtener_embedding_cls(texto):
    inputs = tokenizador(
        texto, return_tensors='pt',
        truncation=True, max_length=128    # ✅ truncar para consistencia
    )
    with torch.no_grad():
        outputs = modelo_bert(**inputs)
    # ✅ Solo el token [CLS] en posición 0 → shape (768,)
    cls = outputs.last_hidden_state[:, 0, :].squeeze(0).numpy()
    return cls

import numpy as np
X = np.array([obtener_embedding_cls(t) for t in textos])  # ✅ shape (N, 768)

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)


---
# BLOQUE 5 – Bayes

## Ejercicio 5.1 – Multiplicar probabilidades sin logaritmos (underflow)
**Caso**: Se implementa Naive Bayes desde cero para clasificar texto.

In [ ]:
# ❌ CÓDIGO CON ERRORES (implementación manual)
import numpy as np

def predecir_bayes_manual(texto, probabilidades_clase, probabilidades_palabra):
    palabras = texto.split()
    mejor_clase = None
    mejor_prob  = -1

    for clase in probabilidades_clase:
        prob = probabilidades_clase[clase]
        for palabra in palabras:
            if palabra in probabilidades_palabra[clase]:
                prob *= probabilidades_palabra[clase][palabra]  # ERROR: underflow
        if prob > mejor_prob:
            mejor_prob  = prob
            mejor_clase = clase
    return mejor_clase

# Para un texto de 100 palabras, prob = 0.001^100 = 10^-300 → underflow a 0.0
# Todos los productos son 0 → no se puede comparar clases


In [ ]:
# ✅ SOLUCIÓN
#
# ERROR: multiplicar probabilidades muy pequeñas genera underflow numérico.
#   Con 100 palabras y p ~ 0.001 por palabra: 0.001^100 = 1e-300 → 0 en float64.
#
# CORRECCIÓN: trabajar en espacio logarítmico.
#   log(a*b*c) = log(a) + log(b) + log(c)
#   Se comparan las log-probabilidades (más negativo = menor probabilidad).

def predecir_bayes_log(texto, log_prob_clase, log_prob_palabra):
    palabras = texto.split()
    mejor_clase    = None
    mejor_log_prob = float('-inf')

    for clase in log_prob_clase:
        log_prob = log_prob_clase[clase]             # log P(clase)
        for palabra in palabras:
            if palabra in log_prob_palabra[clase]:
                log_prob += log_prob_palabra[clase][palabra]  # ✅ suma de logs
        if log_prob > mejor_log_prob:
            mejor_log_prob = log_prob
            mejor_clase    = clase
    return mejor_clase

# GaussianNB de sklearn ya hace esto internamente.


---
## Ejercicio 5.2 – Bayes con feature altamente correlacionados
**Caso**: Clasificar enfermedades usando síntomas médicos muy correlacionados.

In [ ]:
# ❌ CÓDIGO CON PROBLEMAS METODOLÓGICOS
from sklearn.naive_bayes import GaussianNB

# Features: fiebre, temperatura_corporal, temperatura_celsius, temperatura_fahrenheit
# temperatura_corporal, temperatura_celsius y temperatura_fahrenheit son la misma info
X = df[['fiebre', 'temperatura_corporal', 'temperatura_celsius', 'temperatura_fahrenheit',
        'tos', 'congestion']]

bayes = GaussianNB()
bayes.fit(X_train, y_train)
# El modelo asume que todos los features son INDEPENDIENTES dado la clase.
# Tener la temperatura 3 veces triplica artificialmente su peso.


In [ ]:
# ✅ SOLUCIÓN
#
# SUPUESTO NAIVE de Naive Bayes: INDEPENDENCIA CONDICIONAL entre features.
#   P(X1, X2, X3 | Y) = P(X1|Y) * P(X2|Y) * P(X3|Y)
#
# Si los features están altamente correlacionados, se viola este supuesto y
#   el modelo sobrepondera esas variables repetidas.
#
# CORRECCIÓN: eliminar features redundantes o usar PCA antes.

# Eliminar columnas redundantes
X = df[['fiebre', 'temperatura_corporal', 'tos', 'congestion']]  # ✅ solo una temp

# O verificar correlación antes de elegir features:
import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlación entre features – detectar redundancias')
plt.show()


---
## Ejercicio 5.3 – GaussianNB para datos de conteo
**Caso**: Clasificar documentos según frecuencia de palabras (bag of words).

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1000)
X_counts = vectorizer.fit_transform(textos)  # conteos de palabras: 0, 1, 2, 3...

bayes = GaussianNB()   # ERROR: GaussianNB asume distribución Normal continua
bayes.fit(X_counts, y)
# X_counts son enteros no negativos (conteos), no valores continuos normales


In [ ]:
# ✅ SOLUCIÓN
#
# TIPOS DE NAIVE BAYES:
#   GaussianNB:     para features CONTINUAS (asume distribución Normal)
#   MultinomialNB:  para CONTEOS de palabras / frecuencias discretas ← ESTE CASO
#   BernoulliNB:    para features BINARIAS (presencia/ausencia de palabras)
#   ComplementNB:   MultinomialNB mejorado para datasets desbalanceados

from sklearn.naive_bayes import MultinomialNB  # ✅ para conteos

bayes = MultinomialNB(alpha=1.0)  # alpha: suavizado de Laplace (evita prob=0)
bayes.fit(X_counts, y)


---
# BLOQUE 6 – Clustering

## Ejercicio 6.1 – DBSCAN con epsilon demasiado pequeño
**Caso**: Todo sale como ruido (label = -1).

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.001, min_samples=5)   # ERROR: eps casi cero
labels = db.fit_predict(X_scaled)

print(f'Clusters: {len(set(labels)) - (1 if -1 in labels else 0)}')
print(f'Ruido:    {(labels == -1).sum()}')
# Clusters: 0, Ruido: N (todos son outliers porque nadie tiene vecinos a distancia 0.001)


In [ ]:
# ✅ SOLUCIÓN
#
# DBSCAN: parámetro eps
#   eps muy pequeño → nadie tiene vecinos → todo es ruido
#   eps muy grande  → todos son vecinos → un solo cluster gigante
#
# CÓMO CALIBRAR eps:
#   Usar k-distance plot: calcular la distancia al k-ésimo vecino más cercano
#   para cada punto y graficarlas ordenadas. El 'codo' de esa curva sugiere eps.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

k = 5  # mismo que min_samples
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distancias, _ = nn.kneighbors(X_scaled)
distancias_k = np.sort(distancias[:, k-1])[::-1]  # distancia al k-ésimo vecino

plt.plot(distancias_k)
plt.xlabel('Puntos ordenados')
plt.ylabel(f'Distancia al {k}-ésimo vecino')
plt.title('k-distance plot – busca el codo para eps')
plt.grid(True)
plt.show()
# El codo de la curva indica el eps apropiado


---
## Ejercicio 6.2 – Clustering jerárquico: linkage incorrecto
**Caso**: Clustering jerárquico aglomerativo con distancia ward pero matriz de correlación.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import pdist, squareform
import numpy as np

# Se quiere usar distancia de correlación (1 - correlacion)
dist_matrix = 1 - np.corrcoef(X_scaled)   # Matriz de distancia de correlación

# ERROR: linkage='ward' requiere distancias Euclidianas, no puede usar matriz personalizada
modelo = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward',
    affinity='precomputed'   # ward no acepta affinity precomputed
)
labels = modelo.fit_predict(dist_matrix)


In [ ]:
# ✅ SOLUCIÓN
#
# MÉTODOS DE LINKAGE (cómo medir distancia entre clusters):
#   'ward':     minimiza la varianza intra-cluster → SOLO con distancia Euclidiana
#   'complete': max distancia entre puntos de dos clusters → acepta cualquier métrica
#   'average':  promedio de distancias → acepta cualquier métrica
#   'single':   min distancia (tiende a 'chaining') → acepta cualquier métrica
#
# Para usar matriz de distancia propia → NO usar 'ward'

# Opción A: usar linkage='complete' con matriz precomputada
modelo = AgglomerativeClustering(
    n_clusters=4,
    linkage='complete',
    metric='precomputed'    # ✅
)
labels = modelo.fit_predict(dist_matrix)

# Opción B: usar ward con datos originales escalados
modelo_ward = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward'
)
labels_ward = modelo_ward.fit_predict(X_scaled)   # ✅ datos, no matriz de distancia


---
## Ejercicio 6.3 – Interpretar clusters como si fueran clases con etiquetas fijas
**Caso**: Se compara un modelo K-Means con labels del año pasado.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

# El año pasado K-Means asignó: clientes VIP → cluster 0, básico → cluster 1
km = KMeans(n_clusters=2, random_state=99)
labels_nuevo = km.fit_predict(X_nuevo)

# Asumen que cluster 0 sigue siendo VIP
acc = accuracy_score(y_vip_real, labels_nuevo)  # ERROR: clusters no tienen orden fijo
print(f'Accuracy: {acc}')   # puede dar 0% (invertido) aunque el clustering sea perfecto


In [ ]:
# ✅ SOLUCIÓN
#
# Los labels de K-Means son ARBITRARIOS: el cluster 0 este año puede ser el 1 del año pasado.
#   No hay garantía de correspondencia entre ejecuciones.
#
# CORRECCIÓN: nunca comparar labels directamente.
#   Si se tienen etiquetas reales para validar (externas), usar:
#   - Adjusted Rand Index (ARI)
#   - Normalized Mutual Information (NMI)
#   - Silhouette (para calidad interna sin etiquetas)

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(y_vip_real, labels_nuevo)
nmi = normalized_mutual_info_score(y_vip_real, labels_nuevo)
print(f'ARI: {ari:.3f} (1.0 = perfecto, 0 = aleatorio)')
print(f'NMI: {nmi:.3f}')

# Para interpretar qué cluster corresponde a qué grupo:
import pandas as pd
df_resultado = pd.DataFrame({'cluster': labels_nuevo, 'vip_real': y_vip_real})
print(df_resultado.groupby('cluster')['vip_real'].value_counts())


---
# BLOQUE 7 – Métricas y Evaluación

## Ejercicio 7.1 – Interpretar la matriz de confusión al revés
**Caso**: Detección de enfermedades. El equipo médico quiere minimizar los falsos negativos.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
print(cm)
# [[850, 50]
#  [80,  20]]

# El equipo reporta:
# 'Tenemos 850 aciertos y solo 50 errores en casos negativos → el modelo es bueno'
# ERROR: ignoran que hay 80 enfermos no detectados (falsos negativos)


In [ ]:
# ✅ SOLUCIÓN: cómo leer la matriz de confusión
#
#                 Predicho: NEG  Predicho: POS
#  Real: NEG   [[   TN=850,       FP=50   ]]   ← negativos reales
#  Real: POS   [[   FN=80,        TP=20   ]]   ← positivos reales (enfermos)
#
# TN (True Negative):  sanos que el modelo predijo sanos ✅
# FP (False Positive): sanos que el modelo predijo enfermos ❌ (alarma falsa)
# FN (False Negative): ENFERMOS que el modelo predijo sanos ❌ ← LO MÁS PELIGROSO
# TP (True Positive):  enfermos detectados correctamente ✅
#
# MÉTRICAS CLAVE para medicina:
#   Recall (Sensibilidad) = TP / (TP + FN) = 20 / (20 + 80) = 0.20  ← MUY BAJO
#   Un Recall de 0.20 significa que solo detectamos el 20% de enfermos.

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['Sano', 'Enfermo']))
# Recall de 'Enfermo' = 0.20 → alarma: el modelo es muy malo en detectar la clase positiva


---
## Ejercicio 7.2 – F1 macro vs F1 weighted en dataset desbalanceado
**Caso**: Dataset con 3 clases: 1000 ejemplos de A, 1000 de B, solo 20 de C.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.metrics import f1_score

# El modelo es perfecto en A y B, pero nunca predice C
# y_test: 1000*A, 1000*B, 20*C
# y_pred: predice A y B perfectamente, NUNCA predice C

f1_weighted = f1_score(y_test, y_pred, average='weighted')
print(f'F1 Weighted: {f1_weighted:.3f}')  # ~0.99 ← ¡parece excelente!

# El equipo reporta 99% F1 y el cliente aprueba el modelo.
# ERROR: F1 weighted pondera por soporte (número de muestras).
# La clase C tiene solo 20 muestras → su F1=0 no afecta casi el promedio.


In [ ]:
# ✅ SOLUCIÓN
#
# TIPOS DE PROMEDIO EN F1:
#   'micro':    cuenta TP/FP/FN totales (similar a accuracy, afectado por tamaño de clase)
#   'macro':    promedio simple de F1 por clase → todas las clases pesan IGUAL
#   'weighted': promedio ponderado por soporte → favorece clases mayoritarias
#
# Para datasets desbalanceados donde TODAS las clases importan → usar 'macro'
# Para reportar con contexto → reportar classification_report completo

from sklearn.metrics import f1_score, classification_report

print(f'F1 Macro:    {f1_score(y_test, y_pred, average="macro"):.3f}')    # ~0.67
print(f'F1 Weighted: {f1_score(y_test, y_pred, average="weighted"):.3f}') # ~0.99 (engañoso)

print()
print(classification_report(y_test, y_pred, target_names=['A', 'B', 'C']))
# La clase C tiene F1 = 0.00 → el modelo falla completamente en ella


---
## Ejercicio 7.3 – R² para clasificación
**Caso**: Equipo usa R² para evaluar un clasificador binario.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.metrics import r2_score

# Modelo de clasificación binaria
y_pred = model.predict(X_test)          # predicciones: 0 o 1

r2 = r2_score(y_test, y_pred)
print(f'R²: {r2:.3f}')   # Puede dar valores negativos o sin interpretación clara

# ERROR: R² es para REGRESIÓN. No tiene interpretación en clasificación.


In [ ]:
# ✅ SOLUCIÓN
#
# TABLA DE MÉTRICAS CORRECTAS POR TIPO DE PROBLEMA:
#
#   REGRESIÓN:        MSE, RMSE, MAE, R²
#   CLASIFICACIÓN:    Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
#   CLUSTERING:       Silhouette, Inertia, ARI, NMI
#
# Para clasificación binaria:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

y_prob = model.predict_proba(X_test)[:, 1]  # probabilidad de clase positiva
y_pred = (y_prob > 0.5).astype(int)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'F1:        {f1_score(y_test, y_pred):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}')


---
## Ejercicio 7.4 – Ajustar el umbral de clasificación
**Caso**: Detección de fraude donde el costo de no detectar es muy alto.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)   # usa umbral 0.5 por defecto

# Recall de fraude = 0.40 → solo detectamos el 40% de los fraudes
# El banco prefiere más falsas alarmas que dejar pasar fraudes
# ERROR: no se ajusta el umbral


In [ ]:
# ✅ SOLUCIÓN
#
# El umbral 0.5 no siempre es óptimo.
# Para fraude: preferimos Recall alto (detectar más fraudes, aunque haya más falsas alarmas)
# Bajando el umbral → más positivos predichos → Recall sube, Precision baja
#
# Usa la curva Precision-Recall o ROC para elegir el umbral.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

y_prob = clf.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1],    label='Recall')
plt.xlabel('Umbral'); plt.legend(); plt.grid(True)
plt.title('Precision vs Recall por umbral')
plt.show()

# Elegir umbral que da Recall >= 0.80 para fraude
umbral_optimo = thresholds[np.argmax(recalls >= 0.80)]   # ✅
y_pred_ajustado = (y_prob >= umbral_optimo).astype(int)


---
# BLOQUE 8 – Pipeline y Preprocesamiento

## Ejercicio 8.1 – OrdinalEncoder para variable nominal
**Caso**: Variable 'ciudad' con valores: Bogotá, Medellín, Cali, Barranquilla.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.preprocessing import OrdinalEncoder

# 'ciudad' es NOMINAL: no hay orden entre Bogotá, Medellín, Cali
enc = OrdinalEncoder()
df['ciudad_enc'] = enc.fit_transform(df[['ciudad']])
# Resultado: Barranquilla=0, Bogotá=1, Cali=2, Medellín=3

# El modelo aprende que Medellín (3) > Cali (2) > Bogotá (1) → absurdo
# Se introduce un orden artificial que no existe


In [ ]:
# ✅ SOLUCIÓN
#
# TIPOS DE VARIABLES CATEGÓRICAS:
#   Nominal: sin orden (ciudad, color, tipo de producto) → OneHotEncoder
#   Ordinal: con orden natural (bajo/medio/alto, talla XS/S/M/L/XL) → OrdinalEncoder
#
# Para 'ciudad': variable nominal → usar OneHotEncoder

from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ciudad_encoded = enc.fit_transform(df[['ciudad']])
# Resultado: una columna binaria por ciudad
# [1,0,0,0] = Barranquilla, [0,1,0,0] = Bogotá, etc.
# No hay orden implícito ✅

ciudades_df = pd.DataFrame(ciudad_encoded, columns=enc.get_feature_names_out(['ciudad']))


---
## Ejercicio 8.2 – SMOTE antes del train_test_split
**Caso**: Dataset desbalanceado, se aplica SMOTE para balancear.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)   # ERROR: SMOTE en todo el dataset

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2)
# El test set tiene ejemplos sintéticos → evaluación sesgada


In [ ]:
# ✅ SOLUCIÓN
#
# SMOTE genera ejemplos SINTÉTICOS interpolando entre ejemplos reales.
# Si se aplica antes del split, ejemplos sintéticos del train pueden
# terminar en el test set → el test ya no representa datos reales.
#
# ORDEN CORRECTO: split → SMOTE solo en train

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# 1. Split primero
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# 2. SMOTE solo en train
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)  # ✅

# 3. Test set intacto con datos REALES
print(f'Train: {len(X_train_res)} muestras (con sintéticos)')
print(f'Test:  {len(X_test)} muestras (datos reales solamente)')


---
## Ejercicio 8.3 – Imputar valores faltantes antes del split
**Caso**: Dataset con 15% de valores nulos en la columna 'salario'.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.impute import SimpleImputer
import pandas as pd

imputer = SimpleImputer(strategy='median')
df['salario'] = imputer.fit_transform(df[['salario']])  # ERROR: imputa en todo el dataset

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# La mediana del test set ya contaminó la imputación del train


In [ ]:
# ✅ SOLUCIÓN
#
# SimpleImputer aprende la mediana/media/moda de los datos donde hace fit.
# Si hace fit en todo el dataset, la estadística del test contamina el train.
#
# CORRECCIÓN: usar Pipeline → garantiza que fit se hace solo en train.

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ✅ Split primero
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ✅ Pipeline: impute → scale → model (fit solo en train)
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier())
])
pipe.fit(X_train, y_train)    # ← internamente: imputer.fit(X_train), scaler.fit(...)
print(f'Test accuracy: {pipe.score(X_test, y_test):.4f}')


---
## Ejercicio 8.4 – No usar stratify en split con clases desbalanceadas
**Caso**: Dataset: 95% clase 0, 5% clase 1. Sin stratify puede quedar sin clase 1 en test.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
    # ERROR: sin stratify=y
)

# Con mala suerte (o con pocos ejemplos de clase 1):
# y_test puede tener solo clase 0
# → AUC-ROC no se puede calcular
# → El modelo no pudo ver suficientes ejemplos positivos durante entrenamiento


In [ ]:
# ✅ SOLUCIÓN
#
# stratify=y garantiza que la proporción de clases en train y test
# sea igual a la del dataset original.
#
# Con 95%/5%: sin stratify puede quedar 99%/1% en uno y 91%/9% en otro.
# Con stratify: ambos tendrán exactamente 95%/5%.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # ✅ mantiene la distribución de clases
)

import pandas as pd
print('Distribución train:', pd.Series(y_train).value_counts(normalize=True).to_dict())
print('Distribución test: ', pd.Series(y_test).value_counts(normalize=True).to_dict())


---
## Ejercicio 8.5 – Columnas distintas en train y test por OneHotEncoder
**Caso**: Columna 'color' tiene valores: {rojo, azul, verde} en train y {rojo, azul, amarillo} en test.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ERROR: se hace OHE por separado en train y test
enc_train = OneHotEncoder(sparse_output=False)
X_train_enc = enc_train.fit_transform(X_train[['color']])
# Columnas: color_azul, color_rojo, color_verde  (3 columnas)

enc_test = OneHotEncoder(sparse_output=False)
X_test_enc = enc_test.fit_transform(X_test[['color']])
# Columnas: color_amarillo, color_azul, color_rojo  (3 columnas pero DISTINTAS)

# El modelo entrenado con 3 columnas recibe 3 columnas diferentes → resultados incorrectos


In [ ]:
# ✅ SOLUCIÓN
#
# El encoder debe ser el MISMO objeto, fit solo en train, transform en ambos.
# handle_unknown='ignore' maneja categorías nuevas en test (las convierte en all-zeros).

from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # ✅
enc.fit(X_train[['color']])   # ✅ fit solo en train

X_train_enc = enc.transform(X_train[['color']])
X_test_enc  = enc.transform(X_test[['color']])   # ✅ mismas columnas
# 'amarillo' en test → vector de ceros (ignorado), no genera nueva columna

print('Columnas:', enc.get_feature_names_out(['color']))
# ['color_azul', 'color_rojo', 'color_verde'] → consistente en train y test


---
## Ejercicio 8.6 – Escalar variables categóricas codificadas como binarias
**Caso**: La variable 'es_vip' es 0 o 1. Se escala junto con las numéricas.

In [ ]:
# ❌ CÓDIGO CON ERRORES
from sklearn.preprocessing import StandardScaler

# df tiene columnas: 'edad', 'ingresos', 'es_vip' (0/1), 'ciudad_bogota' (0/1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['edad', 'ingresos', 'es_vip', 'ciudad_bogota']])

# Después del escalado: 'es_vip' ya no es 0/1
# pasa a ser algo como -0.23 o 1.87 → pierde su significado binario


In [ ]:
# ✅ SOLUCIÓN
#
# Las variables binarias (0/1) y one-hot NO deben escalarse.
#   Escalarlas distorsiona su significado y puede perjudicar al modelo.
#
# CORRECCIÓN: escalar solo las columnas numéricas continuas.
# Usar ColumnTransformer para aplicar transformaciones selectivas.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

numericas  = ['edad', 'ingresos']             # continuas → escalar
binarias   = ['es_vip', 'ciudad_bogota']      # binarias → NO escalar

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numericas),     # ✅ solo escalamos las continuas
    ('bin', 'passthrough', binarias)          # ✅ pasamos las binarias sin tocar
])

X_preparado = preprocessor.fit_transform(X_train)


---
## Ejercicio 8.7 – Caso completo de parcial: detecta TODOS los errores

**Caso**: Queremos predecir si un cliente va a churn (abandonar el servicio).
Dataset: 90% no-churn, 10% churn. Variables: edad (numérica), plan (nominal: básico/premium/pro), meses_activo (numérica), región (nominal), churn (target: 0/1).

In [ ]:
# ❌ CÓDIGO CON MÚLTIPLES ERRORES – Encuentra todos
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

df = pd.read_csv('clientes.csv')

# --- Preprocesamiento ---
# Imputar
imputer = SimpleImputer(strategy='mean')
df['edad'] = imputer.fit_transform(df[['edad']])             # ERROR A

# Encodear
enc = OrdinalEncoder()
df['plan']   = enc.fit_transform(df[['plan']])               # ERROR B
df['region'] = enc.fit_transform(df[['region']])             # ERROR B (mismo)

# Escalar
scaler = StandardScaler()
X = df.drop('churn', axis=1)
y = df['churn']
X_scaled = scaler.fit_transform(X)                           # ERROR C

# Balancear
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)               # ERROR D

# Split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2)
# (Sin stratify)                                              # ERROR E

# Modelo
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(2, activation='softmax')                    # ERROR F
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1.0),      # ERROR G
    loss='categorical_crossentropy',                         # ERROR H
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=100)                      # (sin early stopping ni val)

y_pred = np.argmax(model.predict(X_test), axis=1)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')         # ERROR I


In [ ]:
# ✅ SOLUCIÓN COMPLETA – Ejercicio 8.7
#
# ERROR A: imputer.fit_transform antes del split → data leakage
#   CORRECCIÓN: split primero, usar Pipeline
#
# ERROR B: OrdinalEncoder para 'plan' y 'region' (variables NOMINALES)
#   'plan' básico/premium/pro tiene un orden natural → OrdinalEncoder podría usarse
#   pero 'region' no tiene orden → OneHotEncoder
#   CORRECCIÓN: OneHotEncoder para nominales sin orden
#
# ERROR C: scaler.fit_transform(X) antes del split → data leakage
#   CORRECCIÓN: split primero, fit scaler solo en X_train
#
# ERROR D: SMOTE antes del split → sintéticos en test set
#   CORRECCIÓN: SMOTE solo en X_train después del split
#
# ERROR E: sin stratify en split con dataset desbalanceado (90/10)
#   CORRECCIÓN: stratify=y
#
# ERROR F: Dense(2, softmax) para clasificación binaria
#   Se puede, pero lo estándar es Dense(1, sigmoid) para binario
#   Si se usa Dense(2), la pérdida debe ser categorical_crossentropy con y one-hot
#   CORRECCIÓN: Dense(1, activation='sigmoid')
#
# ERROR G: learning_rate=1.0 → loss explotará a NaN
#   CORRECCIÓN: lr=0.001 (default de Adam)
#
# ERROR H: categorical_crossentropy con Dense(1, sigmoid)
#   CORRECCIÓN: binary_crossentropy
#
# ERROR I: accuracy en dataset desbalanceado (10% churn)
#   Un modelo que predice siempre 'no churn' tiene 90% accuracy
#   CORRECCIÓN: usar ROC-AUC, F1, recall del churn

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import classification_report, roc_auc_score

# ✅ 1. Split PRIMERO con stratify
X = df.drop('churn', axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # ✅
)

# ✅ 2. Definir columnas por tipo
num_cols = ['edad', 'meses_activo']
ord_cols = ['plan']      # básico < premium < pro → hay orden
nom_cols = ['region']    # sin orden

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())])
ord_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                     ('enc', OrdinalEncoder(categories=[['básico','premium','pro']]))])
nom_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                     ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('ord', ord_pipe, ord_cols),
    ('nom', nom_pipe, nom_cols)
])

# ✅ 3. Transformar: fit en train, transform en test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# ✅ 4. SMOTE solo en train
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

# ✅ 5. Modelo correcto
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_res.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')   # ✅ sigmoid para binario
])
model.compile(
    optimizer=keras.optimizers.Adam(0.001), # ✅
    loss='binary_crossentropy',             # ✅
    metrics=['accuracy']
)

es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_res, y_train_res, epochs=100, validation_split=0.15, callbacks=[es])

# ✅ 6. Evaluar con métricas apropiadas
y_prob = model.predict(X_test_prep).flatten()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['No churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')


---
# RESUMEN FINAL: Errores más peligrosos del parcial

```
Rank 1 – Data Leakage
  scaler / imputer / tokenizer .fit() antes del train_test_split
  SMOTE antes del split
  Validation data = test data

Rank 2 – Activación / Pérdida incorrecta
  relu en salida → sigmoid (binario) o softmax (multiclase)
  mse para clasificación → binary_crossentropy / sparse_categorical_crossentropy
  categorical_crossentropy con etiquetas enteras → sparse_categorical_crossentropy

Rank 3 – CNN sin Flatten o sin normalizar
  Dense antes de Flatten
  input_shape sin canal (28,28) en vez de (28,28,1)
  Imágenes en [0,255] sin dividir entre 255

Rank 4 – NLP: texto crudo a la red
  Sin Tokenizer + pad_sequences + Embedding
  SimpleRNN para secuencias largas → LSTM/GRU
  return_sequences mal configurado al apilar LSTMs

Rank 5 – Clustering sin escalar
  K-Means con datos sin escalar
  Variables categóricas directas en K-Means
  k elegido sin método del codo + silhouette

Rank 6 – Métricas incorrectas
  Accuracy en dataset desbalanceado
  R² para clasificación
  Accuracy para clustering
  F1 weighted oculta clases minoritarias → usar F1 macro
```